# SPAI Inference Results

Objective: present saved SPAI detector outputs for NTIRE and Z-Image-Turbo without constructing the model or dataloaders by default.

Artifacts preserved by this notebook:
- `data/silver/spai/ntire_scores.csv`
- `data/silver/spai/z_image_turbo_scores.csv`
- `data/silver/spai/ntire_metrics.json`
- `data/silver/spai/experiment_config.json`
- `data/silver/spai/plots/`


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import SVG, display
from sklearn.metrics import precision_recall_curve, roc_curve

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent.parent

OUTPUT_DIR = REPO_ROOT / "data" / "silver" / "spai"
PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

NTIRE_SCORES = OUTPUT_DIR / "ntire_scores.csv"
ZIT_SCORES = OUTPUT_DIR / "z_image_turbo_scores.csv"
NTIRE_METRICS = OUTPUT_DIR / "ntire_metrics.json"
EXPERIMENT_CONFIG = OUTPUT_DIR / "experiment_config.json"

for path in [NTIRE_SCORES, ZIT_SCORES, NTIRE_METRICS, EXPERIMENT_CONFIG]:
    if not path.exists():
        raise FileNotFoundError(path)

print(f"Repo root: {REPO_ROOT}")
print(f"SPAI artifacts: {OUTPUT_DIR}")

In [ ]:
results_ntire = pd.read_csv(NTIRE_SCORES)
results_zit = pd.read_csv(ZIT_SCORES)
with NTIRE_METRICS.open() as handle:
    metrics_ntire = json.load(handle)
with EXPERIMENT_CONFIG.open() as handle:
    experiment_config = json.load(handle)

summary_ntire = pd.DataFrame(
    [
        {
            "dataset": "NTIRE",
            "images": len(results_ntire),
            "accuracy": metrics_ntire.get("accuracy"),
            "average_precision": metrics_ntire.get("average_precision"),
            "auc_roc": metrics_ntire.get("auc_roc"),
            "f1": metrics_ntire.get("f1"),
            "precision": metrics_ntire.get("precision"),
            "recall": metrics_ntire.get("recall"),
        }
    ]
)
summary_zit = pd.DataFrame(
    [
        {
            "dataset": "Z-Image-Turbo",
            "images": len(results_zit),
            "mean_score": results_zit["score"].mean(),
            "std_score": results_zit["score"].std(),
            "median_score": results_zit["score"].median(),
            "score_gt_0_5_rate": (results_zit["score"] > 0.5).mean(),
        }
    ]
)

display(summary_ntire.style.format(precision=4))
display(summary_zit.style.format(precision=4))

In [ ]:
y_true = results_ntire["label"].to_numpy()
y_score = results_ntire["score"].to_numpy()

fig, ax = plt.subplots(figsize=(6, 4))
for label, color, name in [(0, "#276FBF", "NTIRE real"), (1, "#C84630", "NTIRE fake")]:
    subset = results_ntire.loc[results_ntire["label"] == label, "score"]
    ax.hist(subset, bins=40, alpha=0.55, label=name, color=color, density=True)
ax.hist(
    results_zit["score"],
    bins=25,
    alpha=0.55,
    label="Z-Image-Turbo fake",
    color="#2A9D8F",
    density=True,
)
ax.axvline(0.5, color="black", linestyle="--", linewidth=0.9)
ax.set_xlabel("Fake probability")
ax.set_ylabel("Density")
ax.set_title("SPAI score distribution")
ax.legend(frameon=False, fontsize=8)
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "score_histogram.svg", format="svg")
plt.close(fig)

fpr, tpr, _ = roc_curve(y_true, y_score)
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(
    fpr, tpr, color="#C84630", linewidth=2, label=f"AUC={metrics_ntire['auc_roc']:.4f}"
)
ax.plot([0, 1], [0, 1], "k--", linewidth=0.9)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("SPAI ROC on NTIRE")
ax.legend(frameon=False)
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "ntire_roc_curve.svg", format="svg")
plt.close(fig)

precision, recall, _ = precision_recall_curve(y_true, y_score)
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(recall, precision, color="#C84630", linewidth=2)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title(f"SPAI PR on NTIRE (AP={metrics_ntire['average_precision']:.4f})")
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "ntire_pr_curve.svg", format="svg")
plt.close(fig)

print(f"Saved SVG plots to {PLOTS_DIR}")
display(SVG(filename=PLOTS_DIR / "score_histogram.svg"))

In [ ]:
display(SVG(filename=PLOTS_DIR / "ntire_roc_curve.svg"))
display(SVG(filename=PLOTS_DIR / "ntire_pr_curve.svg"))

In [ ]:
RUN_INFERENCE = False

if RUN_INFERENCE:
    # Long-running path: prefer the project CLI/background process for full regeneration.
    # Example:
    # uv run python -m diffguard.cli.run_spai_inference
    raise RuntimeError(
        "SPAI inference is intentionally not run inside the notebook. "
        "Use the CLI/background job and keep this notebook for presentation."
    )
else:
    print(
        "Inference cell skipped. Use the SPAI CLI/background job only when regenerating scores."
    )

## Notes

SPAI uses a frequency-restoration-oriented detector and performs better than AIDE on NTIRE in the saved run, while AIDE is more sensitive on Z-Image-Turbo. These outputs feed the gold-layer comparison notebook.


---

## Attention Visualization — Failure-Case Study

Select 6 representative images from the NTIRE test set and export SPAI spectral-context attention masks and overlays for qualitative analysis.


In [ ]:
import shutil

import torch
from spai.config import get_config
from spai.data import data_finetune
from spai.models import build_cls_model
from spai.utils import load_pretrained

# Albumentations 2.x compat
data_finetune.ImageCompressionType = type(
    "ImageCompressionType", (), {"JPEG": "jpeg", "WEBP": "webp"}
)

SPAI_CONFIG = REPO_ROOT / "data" / "submodules" / "spai" / "configs" / "spai.yaml"
CHECKPOINT = REPO_ROOT / "data" / "artifacts" / "checkpoints" / "spai.pth"
NTIRE_DIR = REPO_ROOT / "data" / "bronze" / "ntire" / "test_images"
ATTN_OUTPUT = REPO_ROOT / "data" / "gold" / "spai_attention"
ATTN_OUTPUT.mkdir(parents=True, exist_ok=True)

config = get_config(
    {
        "cfg": str(SPAI_CONFIG),
        "batch_size": 1,
        "test_csv": [
            str(REPO_ROOT / "data" / "silver" / "spai" / "ntire_manifest.csv")
        ],
        "test_csv_root": [str(NTIRE_DIR)],
        "pretrained": str(CHECKPOINT),
        "output": str(ATTN_OUTPUT),
        "tag": "spai_attn",
        "opts": [("MODEL.FEATURE_EXTRACTION_BATCH", "1")],
    }
)

model = build_cls_model(config)
model.cuda()
load_pretrained(
    config,
    model,
    __import__("logging").getLogger("spai_attn"),
    checkpoint_path=CHECKPOINT,
)
model.eval()
print(f"Model loaded. VRAM: {torch.cuda.memory_allocated() / 1e6:.0f} MB")

In [ ]:
# --- Case selection ---
scores = pd.read_csv(REPO_ROOT / "data" / "silver" / "spai" / "ntire_scores.csv")

tp_fake = scores[(scores["label"] == 1) & (scores["prediction"] == 1)].nlargest(
    1, "score"
)
fn_fake = scores[(scores["label"] == 1) & (scores["prediction"] == 0)].nlargest(
    1, "score"
)
fp_real = scores[(scores["label"] == 0) & (scores["prediction"] == 1)].nlargest(
    1, "score"
)
tn_real = scores[(scores["label"] == 0) & (scores["prediction"] == 0)].nsmallest(
    1, "score"
)
borderline = scores.iloc[(scores["score"] - 0.5).abs().argsort()[:1]]

already_selected = set(
    pd.concat([tp_fake, fn_fake, fp_real, tn_real, borderline])["image"]
)
pool = scores[~scores["image"].isin(already_selected)].copy()
pool["dist"] = (pool["score"] - 0.5).abs()
misclassified = pool[pool["label"] != pool["prediction"]]
case6 = (
    misclassified.nlargest(1, "dist")
    if len(misclassified) > 0
    else pool.nsmallest(1, "dist")
)

cases = pd.concat([tp_fake, fn_fake, fp_real, tn_real, borderline, case6]).copy()
cases["case_type"] = [
    "tp_fake",
    "fn_fake",
    "fp_real",
    "tn_real",
    "borderline",
    "near_threshold_misclassified",
]
cases["case_id"] = [f"case_{i:02d}_{ct}" for i, ct in enumerate(cases["case_type"], 1)]

for _, r in cases.iterrows():
    print(
        f"{r['case_id']:35s} | L={r['label']} P={r['prediction']} s={r['score']:.6f} | {r['image']}"
    )

In [ ]:
from PIL import Image
from torchvision import transforms as T

# SPAI uses "positive_0_1" normalization: (img - 0) / 1 = img, so just [0,1] range.
# With ORIGINAL_RESOLUTION=True, images are padded to min 224x224.
to_tensor = T.Compose(
    [
        T.ToTensor(),
        T.Pad(padding=0),  # placeholder — padded below
    ]
)


def prepare_image(image_path: Path, min_size: int = 224) -> torch.Tensor:
    """Load image, pad to min_size, return 1xCxHxW tensor in [0,1]."""
    pil_img = Image.open(image_path).convert("RGB")
    w, h = pil_img.size
    pad_h = max(min_size - h, 0)
    pad_w = max(min_size - w, 0)
    if pad_h > 0 or pad_w > 0:
        left = pad_w // 2
        right = pad_w - left
        top = pad_h // 2
        bottom = pad_h - top
        pil_img = T.functional.pad(pil_img, [left, top, right, bottom], fill=0)
    return T.ToTensor()(pil_img).unsqueeze(0).cuda()  # 1xCxHxW


case_dirs = []
for _, row in cases.iterrows():
    case_name = row["case_id"]
    case_dir = ATTN_OUTPUT / case_name
    case_dir.mkdir(parents=True, exist_ok=True)
    case_dirs.append(case_dir)

    # Copy original image
    src = NTIRE_DIR / row["image"]
    shutil.copy2(src, case_dir / "original.jpg")

    tensor = prepare_image(src)

    # Run attention export
    with torch.no_grad():
        pred, attn_masks = model.forward_arbitrary_resolution_batch_with_export(
            x=[tensor],
            feature_extraction_batch_size=1,
            export_dirs=[case_dir],
            export_image_patches=False,
        )
    score = torch.sigmoid(pred).item()

    # Move exported files from patches_attn/ to case_dir root for cleaner layout
    patches_attn_dir = case_dir / "patches_attn"
    if patches_attn_dir.exists():
        for f in patches_attn_dir.iterdir():
            dest = case_dir / f.name
            if not dest.exists():
                shutil.move(str(f), str(dest))
        try:
            patches_attn_dir.rmdir()
        except OSError:
            pass

    # Save metadata
    meta = {
        "image": row["image"],
        "label": int(row["label"]),
        "prediction": int(row["prediction"]),
        "score": float(row["score"]),
        "forward_score": round(score, 6),
        "source": row["source"],
        "case_type": row["case_type"],
    }
    with open(case_dir / "metadata.json", "w") as f:
        json.dump(meta, f, indent=2)

    torch.cuda.empty_cache()
    print(
        f"[{case_name}] forward_score={score:.6f} original_score={row['score']:.6f} label={row['label']} pred={row['prediction']}"
    )

print("\nAll attention exports complete.")

In [ ]:
# --- Build summary table ---
interpretations = {
    "tp_fake": "High-activation regions should align with GAN/diffusion spectral artifacts. Confirms SPAI detects synthetic frequency patterns.",
    "fn_fake": "Missed fake — attention likely diffused or focused on semantically plausible regions, suggesting SPAI found no spectral anomaly.",
    "fp_real": "Real image flagged as fake — attention may concentrate on high-frequency texture or compression artifacts mistaken for synthesis traces.",
    "tn_real": "Low uniform attention expected — SPAI correctly recognizes natural spectral statistics with no suspicious regions.",
    "borderline": "Score near decision boundary — attention map likely shows mixed activation, indicating ambiguous spectral-context evidence.",
    "near_threshold_misclassified": "Confidently misclassified — attention may be dominated by a single high-activation region that overrides contradictory spectral cues.",
}

summary_rows = []
for _, row in cases.iterrows():
    case_dir = ATTN_OUTPUT / row["case_id"]
    with open(case_dir / "metadata.json") as f:
        meta = json.load(f)
    summary_rows.append(
        {
            "case_id": row["case_id"],
            "case_type": row["case_type"],
            "image": row["image"],
            "label": row["label"],
            "prediction": row["prediction"],
            "spai_score": round(row["score"], 6),
            "interpretation": interpretations.get(row["case_type"], ""),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(ATTN_OUTPUT / "summary.csv", index=False)
display(summary_df)
print(f"\nSummary saved to {ATTN_OUTPUT / 'summary.csv'}")

In [ ]:
# --- Display attention visualizations ---
from IPython.display import Image as IPyImage
from IPython.display import display

for _, row in cases.iterrows():
    case_dir = ATTN_OUTPUT / row["case_id"]
    print(
        f"\n### {row['case_id']} | label={row['label']} pred={row['prediction']} score={row['score']:.4f}"
    )

    # Find the overlay file (name contains the forward score)
    overlay_files = sorted(case_dir.glob("attn_overlay_*.png"))
    mask_files = sorted(case_dir.glob("attn_mask_*.png"))
    colormap_files = sorted(case_dir.glob("attn_mask_colormap_*.png"))

    for label, files in [
        ("Original", [case_dir / "original.jpg"]),
        ("Overlay", overlay_files),
        ("Mask", mask_files),
        ("Colormap", colormap_files),
    ]:
        for f in files:
            if f.exists():
                print(f"  {label}: {f.name}")
                display(IPyImage(filename=str(f), width=400))